# RCATT — Colab Training Notebook
Resource-Configuration-Aware Transformer Tracker · T1 Ablation (agnostic baseline)

**Run cells top-to-bottom once per session.**  After §2 the repo and data are on the local SSD and all subsequent cells are rerunnable without re-cloning.

## §1 — Environment check

In [ ]:
import torch, platform
print('Python :', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('VRAM   :', f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else '-')

## §2 — Clone repo & copy data to local SSD

In [ ]:
import os

REPO_URL = 'https://github.com/coledickerson13/RCATT-sim.git'
REPO_DIR = '/content/RCATT-sim'
DATA_DIR = '/content/data'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -5

In [ ]:
# Copy HDF5 files from Drive to local SSD
# Edit DRIVE_DATA_DIR to match where your h5 files live in Drive.
DRIVE_DATA_DIR = '/content/drive/MyDrive/rcatt_data'  # <-- edit if needed

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_DIR}/train.h5'):
    from google.colab import drive
    drive.mount('/content/drive')
    import time, shutil
    for fname in ('train.h5', 'val.h5'):
        src = f'{DRIVE_DATA_DIR}/{fname}'
        dst = f'{DATA_DIR}/{fname}'
        print(f'Copying {fname} ...', end=' ', flush=True)
        t0 = time.time()
        shutil.copy2(src, dst)
        print(f'{time.time()-t0:.1f}s  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('Data already on local SSD')
    for fname in ('train.h5', 'val.h5'):
        print(f'  {fname}: {os.path.getsize(f"{DATA_DIR}/{fname}")/1e6:.0f} MB')

## §3 — Verify dataloader (must show in-memory version)

In [ ]:
import sys, time, inspect
sys.path.insert(0, REPO_DIR)
from model.dataloader import RCATTDataset, make_dataloader

src = inspect.getsource(RCATTDataset.__init__)
ok  = '_data' in src and 'swmr' not in src
print('In-memory dataloader:', '✅ OK' if ok else '❌ OLD VERSION — re-run §2 (git pull)')

print('Loading datasets into RAM...', end=' ', flush=True)
t0 = time.perf_counter()
train_loader = make_dataloader(f'{DATA_DIR}/train.h5', batch_size=64, shuffle=True)
val_loader   = make_dataloader(f'{DATA_DIR}/val.h5',   batch_size=64, shuffle=False)
elapsed = time.perf_counter() - t0
print(f'done ({elapsed:.1f}s)')
print(f'Train: {len(train_loader.dataset):,} seqs  |  {len(train_loader)} batches/epoch')
print(f'Val  : {len(val_loader.dataset):,} seqs  |  {len(val_loader)} batches/epoch')
if elapsed < 0.3:
    print('\n⚠️  Load time suspiciously fast — dataloader may still be lazy.')

## §4 — Build model

In [ ]:
import random, numpy as np, torch, torch.optim as optim
from model import AgnosticTransformer, compute_loss
from config.sim_config import N_TARGET_MAX

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# T1 hyperparameters
D_MODEL, N_HEADS, N_ENC, N_DEC, DIM_FF, DROPOUT = 64, 4, 2, 2, 128, 0.0
BATCH_SIZE, LR, MAX_EPOCHS, ES_PATIENCE         = 64, 5e-4, 100, 15
GRAD_CLIP, LAMBDA_STATE, LAMBDA_EXIST           = 5.0, 2.0, 1.0

model = AgnosticTransformer(
    N_MAX=N_TARGET_MAX, d_model=D_MODEL, n_heads=N_HEADS,
    n_enc_layers=N_ENC, n_dec_layers=N_DEC, dim_ff=DIM_FF,
    dropout=DROPOUT, use_quality=False, use_context=False,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model   : AgnosticTransformer  ({n_params:,} params)')
print(f'Device  : {device}  ({torch.cuda.get_device_name(0)})')
print(f'AMP     : {scaler.is_enabled()}')

## §5 — Training loop

In [ ]:
import csv, os, time
from tqdm.auto import tqdm

OUT_DIR = f'{REPO_DIR}/out/t1'
os.makedirs(OUT_DIR, exist_ok=True)

# Resume from checkpoint if present
start_epoch = 1
best_val    = float('inf')
es_count    = 0
ckpt_path   = f'{OUT_DIR}/last_model.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt: scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_val    = ckpt.get('best_val', float('inf'))
    es_count    = ckpt.get('es_count', 0)
    print(f'Resumed from epoch {ckpt["epoch"]}  (best_val={best_val:.4f})')

# CSV log
csv_path   = f'{OUT_DIR}/loss_history.csv'
fieldnames = ['epoch','train_loss','train_state','train_exist','val_loss','val_state','val_exist','lr','epoch_s']
csv_exists = os.path.exists(csv_path) and start_epoch > 1
csv_file   = open(csv_path, 'a' if csv_exists else 'w', newline='')
writer     = csv.DictWriter(csv_file, fieldnames=fieldnames)
if not csv_exists: writer.writeheader()

def train_epoch(epoch):
    model.train()
    total = st = ex = n = 0
    for batch in tqdm(train_loader, desc=f'  Ep {epoch:>3d} train', leave=False, unit='b'):
        det    = batch['detections'].to(device, non_blocking=True)
        dv     = batch['det_valid' ].to(device, non_blocking=True)
        states = batch['states'    ].to(device, non_blocking=True)
        tmask  = batch['state_valid'] & batch['visible']
        sids   = batch['state_ids']
        gtnorm = model.norm_states(states)
        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            ps, pl = model(det, dv, contexts=None)
        loss, info = compute_loss(
            pred_states=ps.float(), pred_exist_logits=pl.float(),
            gt_states_norm=gtnorm, target_mask=tmask, state_ids=sids,
            lambda_state=LAMBDA_STATE, lambda_exist=LAMBDA_EXIST,
        )
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); st += info['state_loss']; ex += info['exist_loss']; n += 1
    return total/n, st/n, ex/n

def val_epoch(epoch):
    model.eval()
    total = st = ex = n = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'  Ep {epoch:>3d} val  ', leave=False, unit='b'):
            det    = batch['detections'].to(device, non_blocking=True)
            dv     = batch['det_valid' ].to(device, non_blocking=True)
            states = batch['states'    ].to(device, non_blocking=True)
            tmask  = batch['state_valid'] & batch['visible']
            sids   = batch['state_ids']
            gtnorm = model.norm_states(states)
            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
                ps, pl = model(det, dv, contexts=None)
            loss, info = compute_loss(
                pred_states=ps.float(), pred_exist_logits=pl.float(),
                gt_states_norm=gtnorm, target_mask=tmask, state_ids=sids,
                lambda_state=LAMBDA_STATE, lambda_exist=LAMBDA_EXIST,
            )
            total += loss.item(); st += info['state_loss']; ex += info['exist_loss']; n += 1
    return total/n, st/n, ex/n

print(f"{'Ep':>4}  {'tr_loss':>8}  {'tr_st':>7}  {'tr_ex':>7}  {'vl_loss':>8}  {'vl_st':>7}  {'vl_ex':>7}  {'lr':>8}  {'s/ep':>5}  flag")
print('-' * 88)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_st, tr_ex = train_epoch(epoch)
    vl_loss, vl_st, vl_ex = val_epoch(epoch)
    scheduler.step(vl_loss)
    elapsed = time.time() - t0
    cur_lr  = optimizer.param_groups[0]['lr']

    flag = ''
    if vl_loss < best_val:
        best_val = vl_loss; es_count = 0; flag = '★ best'
        torch.save(model.state_dict(), f'{OUT_DIR}/best_model.pt')
    else:
        es_count += 1
        if es_count >= ES_PATIENCE: flag = 'STOP'

    if epoch % 10 == 0 or flag:
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(), 'best_val': best_val, 'es_count': es_count},
                   ckpt_path)

    writer.writerow({'epoch': epoch, 'train_loss': f'{tr_loss:.5f}',
        'train_state': f'{tr_st:.5f}', 'train_exist': f'{tr_ex:.5f}',
        'val_loss': f'{vl_loss:.5f}', 'val_state': f'{vl_st:.5f}', 'val_exist': f'{vl_ex:.5f}',
        'lr': f'{cur_lr:.2e}', 'epoch_s': f'{elapsed:.1f}'})
    csv_file.flush()

    print(f'{epoch:>4}  {tr_loss:>8.4f}  {tr_st:>7.4f}  {tr_ex:>7.4f}  '
          f'{vl_loss:>8.4f}  {vl_st:>7.4f}  {vl_ex:>7.4f}  {cur_lr:>8.2e}  {elapsed:>5.1f}  {flag}')
    if flag == 'STOP':
        print(f'\nEarly stopping ({ES_PATIENCE} epochs without improvement).')
        break

csv_file.close()
print(f'\nBest val loss : {best_val:.5f}')
print(f'Checkpoints   : {OUT_DIR}/')

## §6 — Plot loss curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{OUT_DIR}/loss_history.csv')
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, cols, title in zip(axes,
        [('train_loss','val_loss'),('train_state','val_state'),('train_exist','val_exist')],
        ['Total loss','State loss','Exist loss']):
    ax.plot(df['epoch'], df[cols[0]], label='train')
    ax.plot(df['epoch'], df[cols[1]], label='val')
    ax.set_title(title); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
best_ep = df.loc[df['val_loss'].idxmin()]
plt.suptitle(f"T1  |  best val_loss={best_ep['val_loss']:.4f} @ epoch {int(best_ep['epoch'])}", fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/loss_curves.png', dpi=120)
plt.show()
print(df.tail(10).to_string(index=False))

## §7 — Save checkpoints back to Drive

In [ ]:
import shutil
DRIVE_OUT = '/content/drive/MyDrive/rcatt_checkpoints/t1'  # <-- edit if needed
os.makedirs(DRIVE_OUT, exist_ok=True)
for fname in ('best_model.pt', 'last_model.pt', 'loss_history.csv', 'loss_curves.png'):
    src = f'{OUT_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_OUT}/{fname}')
        print(f'Saved {fname} → Drive')